# Laguna XS.2 — Global Writeability Atlas
## v7 · Causal necessity vs routing vs gradient accessibility vs actual adaptation

**Fresh notebook.** v7 does not append another phase to v6.

v6 established a strong pilot dissociation:

- a highly causal routed expert can be almost non-plastic;
- routing-selected experts can be much more writable;
- gradient-selected experts can be even more writable;
- participation-frequency forcing does not rescue the causal K=1 expert.

v7 asks the population-level question:

\[
\boxed{
\text{Which pre-training expert property actually predicts later adaptation?}
}
\]

For expert \(E_i\), v7 measures:

\[
C_i=\text{causal necessity}
\]

\[
R_i=\text{routing access}
\]

\[
G_i=\text{gradient accessibility}
\]

\[
A_i=\text{actual matched-budget adaptation}
\]

and estimates:

\[
\rho(C,A),\qquad
\rho(R,A),\qquad
\rho(G,A)
\]

### Experimental design

**Global screen — all 9,984 routed expert locations**

- selection-target supervised routing rate/mass;
- selection-target gradient norm;
- selection-control gradient norm;
- target-minus-control gradient specificity.

The global gradient screen is performed **layer-wise**. Only one layer's
fused expert tensors requires gradients at a time, which keeps the extra
gradient allocation to roughly one MoE layer rather than the whole model.

**Population atlas panel — 72 experts**

- 48 population experts sampled independently of causal/gradient/routing rank;
- 24 sentinel experts covering top gradient, gradient-specific, routing, and
  the important v6 anchors.

For all 72:

- exact fixed-routing causal ablation;
- renormalized ablation;
- bootstrap causal interval;
- precise FP32 gradient norm on target/control;
- target-vs-control gradient cosine;
- one-epoch matched adaptation;
- validation target improvement and target-specific gain.

**Important:** correlations intended to estimate population relationships are
computed on the **48 population-sampled experts only**. Sentinels are for
extremes/winner comparisons and are reported separately.

### Leakage control

The original 50-target + 50-control held-out set is deterministically split:

- **atlas validation:** 25 target + 25 control;
- **final test:** 25 target + 25 control.

The 72-expert adaptation atlas is evaluated on validation only.
Final selector confirmation is evaluated on the untouched final test split.

### Final confirmation

Fresh full-budget training, 3 order seeds, compares:

- panel-best causal selector;
- global routing selector;
- global raw-gradient selector;
- global gradient-specific selector;
- validation-best write expert.

A K={1,2,4,8} gradient-vs-routing budget curve is also included.

All expensive phases checkpoint incrementally and resume automatically.

## 1 — Install dependencies

In [2]:
# Keep the CUDA-enabled PyTorch build supplied by the instance.
%pip -q install -U \
  "transformers==5.14.1" \
  "accelerate>=1.10.0" \
  "huggingface_hub>=0.35.0" \
  safetensors pandas numpy psutil tqdm matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 2 — Runtime tuning for g7e.2xlarge

In [3]:
import os

# 8 vCPU host: leave headroom for Python / I/O.
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["MALLOC_ARENA_MAX"] = "4"

# 64 GiB RAM: conservative checkpoint-loading parallelism.
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "2"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "8"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

# CUDA.
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:512"
)

print("Runtime configured for AWS g7e.2xlarge.")

Runtime configured for AWS g7e.2xlarge.


## 3 — Hardware and storage preflight

In [4]:
import os
import shutil
import platform
from pathlib import Path

import psutil
import torch

ram = psutil.virtual_memory()

print("=== Host ===")
print("Python:", platform.python_version())
print("Logical CPUs:", os.cpu_count())
print(f"RAM total:     {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")

print("\n=== CUDA ===")
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f"This notebook expects exactly one GPU; found {torch.cuda.device_count()}."
    )

props = torch.cuda.get_device_properties(0)

print("GPU:", props.name)
print(f"VRAM: {props.total_memory/2**30:.2f} GiB")
print("Compute capability:", torch.cuda.get_device_capability(0))

if props.total_memory / 2**30 < 88:
    raise RuntimeError("Need a 96GB-class GPU (~89 GiB binary or larger).")

if (os.cpu_count() or 0) < 8:
    print("WARNING: fewer than 8 logical CPUs detected.")

if ram.total / 2**30 < 58:
    print("WARNING: less than a 64-GiB-class host detected.")

roots = [
    Path("/home/ec2-user/workspace"),
    Path("/workspace"),
    Path("/mnt/data"),
    Path("/root"),
    Path("/tmp"),
    Path.cwd(),
]

choices = []
seen_devices = set()

for p in roots:
    try:
        if not p.exists() or not os.access(p, os.W_OK):
            continue
        dev = os.stat(p).st_dev
        if dev in seen_devices:
            continue
        seen_devices.add(dev)
        usage = shutil.disk_usage(p)
        choices.append((usage.free, p, usage))
    except OSError:
        pass

if not choices:
    raise RuntimeError("No writable filesystem found.")

_, WORK_ROOT, disk = max(choices, key=lambda x: x[0])

print("\n=== Storage ===")
print("Work root:", WORK_ROOT)
print(f"Disk free: {disk.free/2**30:.2f} GiB")
print("Hardware preflight: PASS")

=== Host ===
Python: 3.10.12
Logical CPUs: 8
RAM total:     62.27 GiB
RAM available: 60.80 GiB

=== CUDA ===
Torch: 2.13.0+cu130
CUDA build: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 94.97 GiB
Compute capability: (12, 0)

=== Storage ===
Work root: /home/ec2-user/workspace
Disk free: 918.22 GiB
Hardware preflight: PASS


## 4 — Load the experiment CSV

Expected schema:

```text
split,kind,prompt,reference
```

Required minimum counts:

```text
selection / target   50
selection / control  50
train     / target   50
heldout   / target   50
heldout   / control  50
```

Lookup order:

1. `LAGUNA_EXPERIMENT_CSV`
2. `/home/ec2-user/workspace/laguna_frontend_experiment.csv`
3. `/workspace/laguna_frontend_experiment.csv`
4. `/mnt/data/laguna_frontend_experiment.csv`

In [5]:
import pandas as pd
import numpy as np

candidates = []

if os.environ.get("LAGUNA_EXPERIMENT_CSV"):
    candidates.append(
        Path(os.environ["LAGUNA_EXPERIMENT_CSV"]).expanduser()
    )

candidates.extend([
    Path("/home/ec2-user/workspace/laguna_frontend_experiment.csv"),
    Path("/workspace/laguna_frontend_experiment.csv"),
    Path("/mnt/data/laguna_frontend_experiment.csv"),
])

EXPERIMENT_CSV = next(
    (p.resolve() for p in candidates if p.exists()),
    None,
)

if EXPERIMENT_CSV is None:
    raise FileNotFoundError(
        "laguna_frontend_experiment.csv not found. "
        "Set LAGUNA_EXPERIMENT_CSV to the full file path."
    )

experiment_df = pd.read_csv(EXPERIMENT_CSV)

required_cols = {"split", "kind", "prompt", "reference"}
missing_cols = required_cols - set(experiment_df.columns)

if missing_cols:
    raise ValueError(
        f"Experiment CSV missing columns: {sorted(missing_cols)}"
    )

experiment_df["split"] = (
    experiment_df["split"].astype(str).str.lower().str.strip()
)
experiment_df["kind"] = (
    experiment_df["kind"].astype(str).str.lower().str.strip()
)

allowed_splits = {"selection", "train", "heldout"}
allowed_kinds = {"target", "control"}

bad_splits = sorted(set(experiment_df["split"]) - allowed_splits)
bad_kinds = sorted(set(experiment_df["kind"]) - allowed_kinds)

if bad_splits:
    raise ValueError(f"Unsupported split labels: {bad_splits}")

if bad_kinds:
    raise ValueError(f"Unsupported kind labels: {bad_kinds}")

minimums = {
    ("selection", "target"): 50,
    ("selection", "control"): 50,
    ("train", "target"): 50,
    ("heldout", "target"): 50,
    ("heldout", "control"): 50,
}

too_small = []

for (split, kind), minimum in minimums.items():
    actual = len(
        experiment_df[
            (experiment_df["split"] == split)
            & (experiment_df["kind"] == kind)
        ]
    )

    if actual < minimum:
        too_small.append((split, kind, actual, minimum))

if too_small:
    raise RuntimeError(
        "Dataset is below required minimums:\n"
        + "\n".join(
            f"{s}/{k}: {a} < {m}"
            for s, k, a, m in too_small
        )
    )

selection_df = experiment_df[
    experiment_df["split"] == "selection"
].reset_index(drop=True)

train_target_df = experiment_df[
    (experiment_df["split"] == "train")
    & (experiment_df["kind"] == "target")
].reset_index(drop=True)

heldout_df = experiment_df[
    experiment_df["split"] == "heldout"
].reset_index(drop=True)

print("Experiment CSV:", EXPERIMENT_CSV)

display(
    experiment_df.groupby(["split", "kind"])
    .size()
    .rename("count")
    .reset_index()
)

Experiment CSV: /home/ec2-user/workspace/laguna_frontend_experiment.csv


,split,kind,count
0,heldout,control,50
1,heldout,target,50
2,selection,control,50
3,selection,target,50
4,train,target,50


### Split leakage check

In [6]:
selection_prompts = set(selection_df["prompt"].astype(str))
train_prompts = set(train_target_df["prompt"].astype(str))
heldout_prompts = set(heldout_df["prompt"].astype(str))

leaks = {
    "selection_vs_train": selection_prompts & train_prompts,
    "selection_vs_heldout": selection_prompts & heldout_prompts,
    "train_vs_heldout": train_prompts & heldout_prompts,
}

for name, overlap in leaks.items():
    print(name, "overlap:", len(overlap))

if any(leaks.values()):
    raise RuntimeError("Prompt leakage detected across splits.")

print("Split leakage check: PASS")

selection_vs_train overlap: 0
selection_vs_heldout overlap: 0
train_vs_heldout overlap: 0
Split leakage check: PASS


## 5 — Resolve the official BF16 Laguna XS.2 checkpoint

In [7]:
from huggingface_hub import snapshot_download

MODEL_ID = "poolside/Laguna-XS.2"

default_model_path = (
    Path("/home/ec2-user/workspace/models/Laguna-XS.2")
    if Path("/home/ec2-user/workspace").exists()
    else WORK_ROOT / "models" / "Laguna-XS.2"
)

MODEL_PATH = Path(
    os.environ.get("LAGUNA_BF16_PATH", str(default_model_path))
).expanduser().resolve()

EXPECTED_SHARDS = [
    f"model-{i:05d}-of-00014.safetensors"
    for i in range(1, 15)
]

complete = (
    (MODEL_PATH / "config.json").exists()
    and all((MODEL_PATH / x).exists() for x in EXPECTED_SHARDS)
)

if not complete:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)

    free_gib = shutil.disk_usage(MODEL_PATH).free / 2**30

    if free_gib < 85:
        raise RuntimeError(
            f"Need ~85 GiB free for a fresh BF16 download; found {free_gib:.1f} GiB."
        )

    print("Downloading Laguna XS.2 BF16...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_PATH),
        allow_patterns=[
            "*.safetensors",
            "*.json",
            "*.py",
            "*.jinja",
            "LICENSE*",
            "README*",
        ],
        max_workers=2,
    )

missing = [x for x in EXPECTED_SHARDS if not (MODEL_PATH / x).exists()]

if missing:
    raise RuntimeError(f"Incomplete checkpoint; missing: {missing}")

sizes = [(x, (MODEL_PATH / x).stat().st_size) for x in EXPECTED_SHARDS]

print("MODEL_PATH:", MODEL_PATH)
print(f"14-shard payload: {sum(n for _, n in sizes)/1e9:.3f} GB")
print("Checkpoint verification: PASS")

MODEL_PATH: /home/ec2-user/workspace/models/Laguna-XS.2
14-shard payload: 66.889 GB
Checkpoint verification: PASS


/home/ec2-user/workspace/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 6 — Register Laguna checkpoint conversion mapping

In [8]:
from transformers.conversion_mapping import (
    get_checkpoint_conversion_mapping,
    register_checkpoint_conversion_mapping,
    USER_REGISTERED_MAPPINGS,
)

laguna_mapping = get_checkpoint_conversion_mapping("laguna")

if laguna_mapping is None:
    raise RuntimeError(
        "Transformers does not expose the native Laguna checkpoint conversion mapping."
    )

register_checkpoint_conversion_mapping(
    "laguna",
    laguna_mapping,
    overwrite=True,
)

if "laguna" not in USER_REGISTERED_MAPPINGS:
    raise RuntimeError("Laguna conversion mapping registration failed.")

print("Laguna checkpoint conversion mapping: REGISTERED")
print("Conversion operations:", len(laguna_mapping))

Laguna checkpoint conversion mapping: REGISTERED
Conversion operations: 4


## 7 — Load BF16 model directly onto the RTX PRO 6000

In [9]:
import gc
import time
import torch
import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(min(6, os.cpu_count() or 6))

try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    output_loading_info=True,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad_(False)

missing = list(loading_info.get("missing_keys", []))
unexpected = list(loading_info.get("unexpected_keys", []))
mismatched = list(loading_info.get("mismatched_keys", []))

critical = [
    k for k in (missing + unexpected)
    if (
        ".mlp.experts" in k
        or ".mlp.gate" in k
        or "e_score_correction_bias" in k
    )
]

if critical or mismatched:
    raise RuntimeError(
        "Critical checkpoint mismatch.\n"
        f"critical sample: {critical[:12]}\n"
        f"mismatched sample: {mismatched[:12]}"
    )

del loading_info
gc.collect()
torch.cuda.synchronize()

free_b, total_b = torch.cuda.mem_get_info()

print(f"Loaded in {(time.time()-t0)/60:.2f} min")
print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/2**30:.2f} GiB")
print(f"GPU peak:      {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free:   {free_b/2**30:.2f} GiB")
print(f"Host RAM available: {psutil.virtual_memory().available/2**30:.2f} GiB")
print("BF16 load: PASS")

Transformers: 5.14.1


Loading weights: 100%|███████████████████████| 639/639 [08:26<00:00,  1.26it/s]


Loaded in 8.51 min
GPU allocated: 62.29 GiB
GPU reserved:  82.26 GiB
GPU peak:      63.29 GiB
Driver free:   12.17 GiB
Host RAM available: 58.74 GiB
BF16 load: PASS


## 8 — Validate Laguna MoE architecture

In [10]:
cfg = model.config

SPARSE_LAYERS = []

for idx, layer in enumerate(model.model.layers):
    mlp = getattr(layer, "mlp", None)

    if (
        mlp is not None
        and hasattr(mlp, "gate")
        and hasattr(mlp, "experts")
        and hasattr(mlp.experts, "gate_up_proj")
        and hasattr(mlp.experts, "down_proj")
    ):
        SPARSE_LAYERS.append(idx)

assert cfg.hidden_size == 2048
assert cfg.num_hidden_layers == 40
assert cfg.num_experts == 256
assert cfg.num_experts_per_tok == 8
assert cfg.moe_intermediate_size == 512
assert len(SPARSE_LAYERS) == 39

sample_mlp = model.model.layers[SPARSE_LAYERS[0]].mlp

assert tuple(sample_mlp.experts.gate_up_proj.shape) == (256, 1024, 2048)
assert tuple(sample_mlp.experts.down_proj.shape) == (256, 2048, 512)

params_per_expert = (
    sample_mlp.experts.gate_up_proj[0].numel()
    + sample_mlp.experts.down_proj[0].numel()
)

print("Sparse layers:", SPARSE_LAYERS)
print(f"Params / expert: {params_per_expert:,} ({params_per_expert/1e6:.3f}M)")
print("Architecture validation: PASS")

Sparse layers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
Params / expert: 3,145,728 (3.146M)
Architecture validation: PASS


## 9 — Correct Laguna teacher-forcing format

The no-thinking assistant prefix is followed by a **newline** before the
reference answer:

```text
<assistant>
</think>
REFERENCE
```

In [11]:
def chat_prefix_text(prompt):
    messages = [{"role": "user", "content": prompt}]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def parse_case(prompt, reference):
    prefix_text = chat_prefix_text(prompt)

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    start = 0

    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise RuntimeError(
            f"Could not identify answer boundary for: {prompt!r}"
        )

    return full_ids, start

## 10 — Generic aligned scoring batch

In [12]:
def build_scoring_batch(df):
    df = df.reset_index(drop=True).copy()

    parsed = [
        parse_case(r.prompt, r.reference)
        for r in df.itertuples(index=False)
    ]

    fulls = [x[0] for x in parsed]
    starts = [x[1] for x in parsed]
    answers = [full[start:] for full, start in zip(fulls, starts)]

    max_prefix = max(starts)
    max_ref = max(len(x) for x in answers)
    batch_size = len(df)
    seq_len = max_prefix + max_ref

    input_ids = torch.full(
        (batch_size, seq_len),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (batch_size, seq_len),
        dtype=torch.long,
    )

    targets = torch.full(
        (batch_size, max_ref),
        -100,
        dtype=torch.long,
    )

    for b, (full, start, ans) in enumerate(zip(fulls, starts, answers)):
        prefix = full[:start]
        prefix_start = max_prefix - len(prefix)

        input_ids[b, prefix_start:max_prefix] = torch.tensor(prefix)
        attention_mask[b, prefix_start:max_prefix] = 1

        input_ids[b, max_prefix:max_prefix+len(ans)] = torch.tensor(ans)
        attention_mask[b, max_prefix:max_prefix+len(ans)] = 1

        targets[b, :len(ans)] = torch.tensor(ans)

    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.clamp_(min=0)

    pred_positions = torch.arange(
        max_prefix - 1,
        max_prefix + max_ref - 1,
        dtype=torch.long,
    )

    return {
        "df": df,
        "input_ids": input_ids.to("cuda:0", non_blocking=True),
        "attention_mask": attention_mask.to("cuda:0", non_blocking=True),
        "position_ids": position_ids.to("cuda:0", non_blocking=True),
        "targets": targets.to("cuda:0", non_blocking=True),
        "pred_positions": pred_positions.to("cuda:0", non_blocking=True),
    }

@torch.inference_mode()
def score_batch(batch):
    import torch.nn.functional as F

    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    per_example = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    result = per_example.detach().cpu().numpy()

    del out, logits, losses, per_example

    return result

## 11 — Build selection and held-out batches

## 11A — Split the held-out set into atlas-validation and untouched final-test

In [ ]:
SPLIT_SEED = 2027
_split_rng = np.random.default_rng(SPLIT_SEED)

validation_parts = []
test_parts = []

for kind in ("target", "control"):
    part = heldout_df[
        heldout_df["kind"] == kind
    ].reset_index(drop=True)

    idx = _split_rng.permutation(len(part))
    midpoint = len(part) // 2

    validation_parts.append(
        part.iloc[idx[:midpoint]]
    )
    test_parts.append(
        part.iloc[idx[midpoint:]]
    )

validation_df = pd.concat(
    validation_parts,
    ignore_index=True,
)

final_test_df = pd.concat(
    test_parts,
    ignore_index=True,
)

assert len(validation_df[validation_df["kind"] == "target"]) == 25
assert len(validation_df[validation_df["kind"] == "control"]) == 25
assert len(final_test_df[final_test_df["kind"] == "target"]) == 25
assert len(final_test_df[final_test_df["kind"] == "control"]) == 25

assert not (
    set(validation_df["prompt"])
    & set(final_test_df["prompt"])
)

print("Atlas validation:")
display(validation_df.groupby("kind").size().rename("count"))

print("Final test:")
display(final_test_df.groupby("kind").size().rename("count"))

## 11B — Results directory and resumable phase files

In [ ]:
RESULTS = WORK_ROOT / "laguna_xs2_v7_writeability_atlas"
RESULTS.mkdir(parents=True, exist_ok=True)

GLOBAL_GRAD_CSV = RESULTS / "global_gradient_screen.csv"
GLOBAL_ROUTE_CSV = RESULTS / "global_routing_screen.csv"
PANEL_CSV = RESULTS / "atlas_panel.csv"
PANEL_CAUSAL_CSV = RESULTS / "panel_causal_scores.csv"
PANEL_GRAD_CSV = RESULTS / "panel_precise_gradients.csv"
PANEL_ADAPT_CSV = RESULTS / "panel_adaptation_validation.csv"
CORRELATION_CSV = RESULTS / "population_correlations.csv"
CONFIRM_CSV = RESULTS / "final_confirmation.csv"
BUDGET_CSV = RESULTS / "budget_curve.csv"

print("Results:", RESULTS)

## 12 — Build selection, validation and final-test scoring batches

In [ ]:
SELECTION_BATCH = build_scoring_batch(selection_df)
VALIDATION_BATCH = build_scoring_batch(validation_df)
FINAL_TEST_BATCH = build_scoring_batch(final_test_df)

SELECTION_BASE_NLL = score_batch(SELECTION_BATCH)
VALIDATION_BASE_NLL = score_batch(VALIDATION_BATCH)
FINAL_TEST_BASE_NLL = score_batch(FINAL_TEST_BATCH)

selection_target_mask = selection_df["kind"].values == "target"
selection_control_mask = selection_df["kind"].values == "control"

validation_target_mask = validation_df["kind"].values == "target"
validation_control_mask = validation_df["kind"].values == "control"

final_test_target_mask = final_test_df["kind"].values == "target"
final_test_control_mask = final_test_df["kind"].values == "control"

print(
    "Selection baseline:",
    "target", float(SELECTION_BASE_NLL[selection_target_mask].mean()),
    "control", float(SELECTION_BASE_NLL[selection_control_mask].mean()),
)

print(
    "Validation baseline:",
    "target", float(VALIDATION_BASE_NLL[validation_target_mask].mean()),
    "control", float(VALIDATION_BASE_NLL[validation_control_mask].mean()),
)

print(
    "Final-test baseline:",
    "target", float(FINAL_TEST_BASE_NLL[final_test_target_mask].mean()),
    "control", float(FINAL_TEST_BASE_NLL[final_test_control_mask].mean()),
)

## 13 — Scoring sanity check

In [ ]:
demo_prefix = chat_prefix_text(selection_df.iloc[0]["prompt"])

print("Prefix tail:", repr(demo_prefix[-60:]))
print(
    "Teacher-forced boundary:",
    repr(
        (
            demo_prefix
            + "\n"
            + selection_df.iloc[0]["reference"]
        )[-90:]
    )
)

b = 0
ids = SELECTION_BATCH["input_ids"][b:b+1]
mask = SELECTION_BATCH["attention_mask"][b:b+1]
pos = SELECTION_BATCH["position_ids"][b:b+1]
keep = SELECTION_BATCH["pred_positions"]

with torch.inference_mode():
    selected_logits = model(
        input_ids=ids,
        attention_mask=mask,
        position_ids=pos,
        use_cache=False,
        logits_to_keep=keep,
        return_dict=True,
    ).logits.float()

    full_logits = model(
        input_ids=ids,
        attention_mask=mask,
        position_ids=pos,
        use_cache=False,
        logits_to_keep=0,
        return_dict=True,
    ).logits[:, keep, :].float()

max_diff = float(
    (selected_logits - full_logits).abs().max().item()
)

print("max |selective - full sliced logits|:", max_diff)

if max_diff > 1e-4:
    raise RuntimeError(
        "Selective-logit scorer does not match full-logit scoring."
    )

del selected_logits, full_logits
torch.cuda.empty_cache()

print("Scoring sanity: PASS")

# Phase A — Exact causal intervention machinery

In [15]:
from contextlib import contextmanager
import types

def get_sparse_mlp(layer_idx):
    layer_idx = int(layer_idx)

    if layer_idx not in SPARSE_LAYERS:
        raise ValueError(f"Layer {layer_idx} is not sparse.")

    return model.model.layers[layer_idx].mlp

@contextmanager
def gate_intervention(
    layer_idx,
    expert_ids=None,
    zero_all_routed=False,
    renormalize=False,
):
    mlp = get_sparse_mlp(layer_idx)
    gate = mlp.gate
    original_forward = gate.forward

    expert_ids = (
        []
        if expert_ids is None
        else [int(x) for x in expert_ids]
    )

    def patched_forward(self, hidden_states):
        router_logits, routing_weights, selected_experts = (
            original_forward(hidden_states)
        )

        if zero_all_routed:
            routing_weights = torch.zeros_like(routing_weights)

        elif expert_ids:
            ids = torch.tensor(
                expert_ids,
                device=selected_experts.device,
                dtype=selected_experts.dtype,
            )

            keep = ~torch.isin(selected_experts, ids)

            routing_weights = (
                routing_weights
                * keep.to(routing_weights.dtype)
            )

            if renormalize:
                denom = routing_weights.sum(
                    dim=-1,
                    keepdim=True,
                )

                routing_weights = torch.where(
                    denom > 0,
                    routing_weights
                    / denom.clamp_min(1e-12),
                    routing_weights,
                )

        return (
            router_logits,
            routing_weights,
            selected_experts,
        )

    gate.forward = types.MethodType(
        patched_forward,
        gate,
    )

    try:
        yield
    finally:
        gate.forward = original_forward

In [16]:
CONTROL_PENALTY = 0.75

def summarize_selection_delta(ablated_nll):
    delta = np.asarray(ablated_nll) - SELECTION_BASE_NLL

    target_delta = float(
        delta[selection_target_mask].mean()
    )

    control_delta = float(
        delta[selection_control_mask].mean()
    )

    causal_specificity = (
        target_delta
        - CONTROL_PENALTY
        * max(control_delta, 0.0)
    )

    return {
        "target_delta_nll": target_delta,
        "control_delta_nll": control_delta,
        "causal_specificity": causal_specificity,
        "per_example_delta": delta,
    }

In [ ]:
def intervention_score(
    layer_idx,
    expert_ids,
    renormalize=False,
):
    with gate_intervention(
        layer_idx,
        expert_ids=expert_ids,
        renormalize=renormalize,
    ):
        nll = score_batch(SELECTION_BATCH)

    return summarize_selection_delta(nll)

def bootstrap_specificity(
    per_example_delta,
    n_boot=5000,
    seed=123,
):
    rng = np.random.default_rng(seed)

    d = np.asarray(
        per_example_delta,
        dtype=np.float64,
    )

    t = d[selection_target_mask]
    c = d[selection_control_mask]

    vals = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for i in range(n_boot):
        tb = rng.choice(
            t,
            size=len(t),
            replace=True,
        ).mean()

        cb = rng.choice(
            c,
            size=len(c),
            replace=True,
        ).mean()

        vals[i] = (
            tb
            - CONTROL_PENALTY
            * max(cb, 0.0)
        )

    return {
        "ci_2.5": float(np.quantile(vals, 0.025)),
        "ci_97.5": float(np.quantile(vals, 0.975)),
        "p_positive": float((vals > 0).mean()),
    }

# Phase B — Teacher-forced training cases

In [33]:
def make_training_case(
    prompt,
    reference,
    max_length=1024,
):
    prefix_text = chat_prefix_text(
        prompt
    )

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    if len(full_ids) > max_length:
        raise ValueError(
            f"{len(full_ids)} tokens > {max_length}"
        )

    start = 0

    for a, b in zip(
        prefix_ids,
        full_ids,
    ):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise ValueError(
            "Could not identify answer boundary."
        )

    input_ids = torch.tensor(
        full_ids,
        dtype=torch.long,
        device="cuda:0",
    ).unsqueeze(0)

    attention_mask = torch.ones_like(
        input_ids
    )

    pred_positions = torch.arange(
        start - 1,
        len(full_ids) - 1,
        dtype=torch.long,
        device="cuda:0",
    )

    targets = torch.tensor(
        full_ids[start:],
        dtype=torch.long,
        device="cuda:0",
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pred_positions": pred_positions,
        "targets": targets,
    }

MAX_TRAIN_TOKENS = 1024

TRAIN_CASES = [
    make_training_case(
        r.prompt,
        r.reference,
        MAX_TRAIN_TOKENS,
    )
    for r in train_target_df.itertuples(
        index=False
    )
]

print("Training examples:", len(TRAIN_CASES))
print(
    "Token lengths:",
    min(
        x["input_ids"].shape[1]
        for x in TRAIN_CASES
    ),
    "to",
    max(
        x["input_ids"].shape[1]
        for x in TRAIN_CASES
    ),
)

Training examples: 50
Token lengths: 66 to 83


In [ ]:
SELECTION_TARGET_CASES = [
    make_training_case(
        r.prompt,
        r.reference,
        1024,
    )
    for r in selection_df[
        selection_df["kind"] == "target"
    ].itertuples(index=False)
]

SELECTION_CONTROL_CASES = [
    make_training_case(
        r.prompt,
        r.reference,
        1024,
    )
    for r in selection_df[
        selection_df["kind"] == "control"
    ].itertuples(index=False)
]

TRAIN_CASES = [
    make_training_case(
        r.prompt,
        r.reference,
        1024,
    )
    for r in train_target_df.itertuples(index=False)
]

print("selection target cases:", len(SELECTION_TARGET_CASES))
print("selection control cases:", len(SELECTION_CONTROL_CASES))
print("train target cases:", len(TRAIN_CASES))

# Phase C — Global supervised routing atlas over all 9,984 expert positions

In [ ]:
def make_supervised_position_mask(batch):
    mask = torch.zeros_like(
        batch["attention_mask"],
        dtype=torch.bool,
    )

    valid_targets = batch["targets"].ne(-100)
    positions = batch["pred_positions"]

    for b in range(mask.shape[0]):
        valid = valid_targets[b]
        pos = positions[valid]
        mask[b, pos] = True

    return mask

@contextmanager
def capture_routing_with_mask(batch, token_mask):
    records = {}
    originals = []

    token_mask_flat = token_mask.reshape(-1).bool()

    for layer_idx in SPARSE_LAYERS:
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward
        originals.append((gate, original))

        def make_forward(idx, original_forward):
            def patched(self, hidden_states):
                logits, weights, selected = original_forward(hidden_states)

                with torch.no_grad():
                    mask = token_mask_flat

                    if mask.numel() != selected.shape[0]:
                        raise RuntimeError(
                            f"Routing mask/token mismatch at layer {idx}: "
                            f"{mask.numel()} vs {selected.shape[0]}"
                        )

                    mask = mask.to(selected.device)

                    ids = selected[mask].reshape(-1).long()
                    ws = weights[mask].reshape(-1).float()

                    counts = torch.bincount(
                        ids,
                        minlength=cfg.num_experts,
                    )

                    wsum = torch.zeros(
                        cfg.num_experts,
                        device=ws.device,
                        dtype=torch.float32,
                    )
                    wsum.scatter_add_(0, ids, ws)

                    records[int(idx)] = {
                        "tokens": int(mask.sum().item()),
                        "counts": counts.cpu(),
                        "weight_sums": wsum.cpu(),
                    }

                return logits, weights, selected

            return patched

        gate.forward = types.MethodType(
            make_forward(layer_idx, original),
            gate,
        )

    try:
        yield records
    finally:
        for gate, original in originals:
            gate.forward = original

In [ ]:
selection_target_df = selection_df[
    selection_df["kind"] == "target"
].reset_index(drop=True)

SELECTION_TARGET_BATCH = build_scoring_batch(
    selection_target_df
)

supervised_mask = make_supervised_position_mask(
    SELECTION_TARGET_BATCH
)

with capture_routing_with_mask(
    SELECTION_TARGET_BATCH,
    supervised_mask,
) as routing_records:
    _ = score_batch(SELECTION_TARGET_BATCH)

routing_rows = []

for layer_idx in SPARSE_LAYERS:
    rec = routing_records[int(layer_idx)]
    tokens = max(1, rec["tokens"])

    for expert_id in range(cfg.num_experts):
        routing_rows.append({
            "layer": int(layer_idx),
            "expert": int(expert_id),
            "selection_supervised_positions": int(rec["tokens"]),
            "selection_supervised_selected_hits": int(
                rec["counts"][expert_id].item()
            ),
            "selection_supervised_selected_rate": float(
                rec["counts"][expert_id].item()
            ) / tokens,
            "selection_supervised_routing_mass": float(
                rec["weight_sums"][expert_id].item()
            ) / tokens,
        })

global_routing_df = pd.DataFrame(routing_rows).sort_values(
    [
        "selection_supervised_routing_mass",
        "selection_supervised_selected_rate",
    ],
    ascending=False,
).reset_index(drop=True)

global_routing_df["routing_rank"] = (
    np.arange(len(global_routing_df)) + 1
)

global_routing_df.to_csv(
    GLOBAL_ROUTE_CSV,
    index=False,
)

GLOBAL_ROUTING_PAIR = tuple(
    global_routing_df.iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

print("GLOBAL_ROUTING_PAIR =", GLOBAL_ROUTING_PAIR)
display(global_routing_df.head(20))

# Phase D — Global layer-wise gradient screen

This is the key new v7 phase.

Instead of instantiating 9,984 separate FP32 expert banks, v7 temporarily
enables gradients for the **fused expert tensors of one sparse layer at a
time**.

For each layer:

1. all other model parameters remain frozen;
2. `gate_up_proj[layer]` and `down_proj[layer]` require gradients;
3. a small fixed selection-target probe is backpropagated;
4. gradient L2 is extracted separately for each of the 256 experts;
5. gradients are cleared;
6. the same is repeated on controls;
7. the layer is returned to frozen state.

Only one layer's full expert gradient exists at a time.

The global scan uses BF16 gradients as an efficient **screening statistic**.
The 72-expert panel is then remeasured with the exact FP32 delta bank.

In [ ]:
GLOBAL_GRAD_PROBE_CASES = 8
GLOBAL_GRAD_CHUNK = 8
RUN_GLOBAL_GRADIENT_SCAN = True

def _fused_expert_grad_norms(
    gate_up_grad,
    down_grad,
    chunk_size=8,
):
    if gate_up_grad is None or down_grad is None:
        raise RuntimeError(
            "Expected fused expert gradients, got None."
        )

    n = gate_up_grad.shape[0]
    out = torch.empty(
        n,
        dtype=torch.float64,
        device="cpu",
    )

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)

        gu = gate_up_grad[start:end].float()
        down = down_grad[start:end].float()

        sq = (
            gu.square().sum(dim=(1, 2))
            + down.square().sum(dim=(1, 2))
        )

        out[start:end] = (
            torch.sqrt(sq)
            .double()
            .cpu()
        )

        del gu, down, sq

    return out.numpy()

def _gradient_objective_for_cases(
    cases,
    max_cases,
):
    used = min(
        int(max_cases),
        len(cases),
    )

    if used <= 0:
        raise RuntimeError("No gradient probe cases.")

    for case in cases[:used]:
        with torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ):
            out = model(
                input_ids=case["input_ids"],
                attention_mask=case["attention_mask"],
                use_cache=False,
                logits_to_keep=case["pred_positions"],
                return_dict=True,
            )

            logits = out.logits.float()

            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                case["targets"].reshape(-1),
            ) / used

        loss.backward()

        del out, logits, loss

def scan_layer_fused_gradients(
    layer_idx,
    target_cases,
    control_cases,
    max_cases=8,
):
    mlp = get_sparse_mlp(layer_idx)
    experts = mlp.experts

    gate_up = experts.gate_up_proj
    down = experts.down_proj

    if gate_up.dtype != torch.bfloat16 or down.dtype != torch.bfloat16:
        print(
            "WARNING: fused expert dtype is",
            gate_up.dtype,
            down.dtype,
        )

    for p in model.parameters():
        p.requires_grad_(False)

    gate_up.requires_grad_(True)
    down.requires_grad_(True)

    model.eval()

    try:
        gate_up.grad = None
        down.grad = None

        _gradient_objective_for_cases(
            target_cases,
            max_cases,
        )

        target_norm = _fused_expert_grad_norms(
            gate_up.grad,
            down.grad,
            GLOBAL_GRAD_CHUNK,
        )

        gate_up.grad = None
        down.grad = None
        torch.cuda.empty_cache()

        _gradient_objective_for_cases(
            control_cases,
            max_cases,
        )

        control_norm = _fused_expert_grad_norms(
            gate_up.grad,
            down.grad,
            GLOBAL_GRAD_CHUNK,
        )

        return target_norm, control_norm

    finally:
        gate_up.grad = None
        down.grad = None

        gate_up.requires_grad_(False)
        down.requires_grad_(False)

        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
if RUN_GLOBAL_GRADIENT_SCAN:
    if GLOBAL_GRAD_CSV.exists():
        global_grad_partial = pd.read_csv(
            GLOBAL_GRAD_CSV
        )
    else:
        global_grad_partial = pd.DataFrame()

    completed_layers = set()

    if not global_grad_partial.empty:
        counts = global_grad_partial.groupby(
            "layer"
        ).size()

        completed_layers = set(
            int(layer)
            for layer, count in counts.items()
            if int(count) == cfg.num_experts
        )

    print(
        "Resuming global gradient scan.",
        "Completed layers:",
        len(completed_layers),
        "/",
        len(SPARSE_LAYERS),
    )

    for layer_idx in tqdm(
        SPARSE_LAYERS,
        desc="Global gradient layers",
    ):
        if int(layer_idx) in completed_layers:
            continue

        torch.cuda.reset_peak_memory_stats()

        target_norm, control_norm = (
            scan_layer_fused_gradients(
                int(layer_idx),
                SELECTION_TARGET_CASES,
                SELECTION_CONTROL_CASES,
                max_cases=GLOBAL_GRAD_PROBE_CASES,
            )
        )

        rows = []

        for expert_id in range(
            cfg.num_experts
        ):
            gt = float(target_norm[expert_id])
            gc_ = float(control_norm[expert_id])

            rows.append({
                "layer": int(layer_idx),
                "expert": int(expert_id),
                "global_target_grad_bf16": gt,
                "global_control_grad_bf16": gc_,
                "global_grad_specific_bf16": (
                    gt
                    - CONTROL_PENALTY * gc_
                ),
                "global_grad_ratio_bf16": (
                    gt / (gc_ + 1e-12)
                ),
                "scan_peak_gpu_gib": float(
                    torch.cuda.max_memory_allocated()
                    / 2**30
                ),
            })

        if global_grad_partial.empty:
            global_grad_partial = pd.DataFrame(
                rows
            )
        else:
            global_grad_partial = pd.concat(
                [
                    global_grad_partial[
                        global_grad_partial["layer"]
                        != int(layer_idx)
                    ],
                    pd.DataFrame(rows),
                ],
                ignore_index=True,
            )

        global_grad_partial.to_csv(
            GLOBAL_GRAD_CSV,
            index=False,
        )

        print(
            f"L{layer_idx}: "
            f"max target G={target_norm.max():.4f}, "
            f"peak={torch.cuda.max_memory_allocated()/2**30:.2f} GiB"
        )

global_gradient_df = pd.read_csv(
    GLOBAL_GRAD_CSV
)

expected_positions = (
    len(SPARSE_LAYERS)
    * cfg.num_experts
)

if len(global_gradient_df) != expected_positions:
    raise RuntimeError(
        f"Global gradient screen incomplete: "
        f"{len(global_gradient_df)} != {expected_positions}"
    )

print(
    "Global gradient screen complete:",
    len(global_gradient_df),
    "expert positions",
)

In [ ]:
GLOBAL_SCREEN = (
    global_gradient_df
    .merge(
        global_routing_df,
        on=["layer", "expert"],
        how="inner",
    )
)

GLOBAL_SCREEN["global_target_grad_rank"] = (
    GLOBAL_SCREEN[
        "global_target_grad_bf16"
    ]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

GLOBAL_SCREEN["global_grad_specific_rank"] = (
    GLOBAL_SCREEN[
        "global_grad_specific_bf16"
    ]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

GLOBAL_SCREEN["global_grad_ratio_rank"] = (
    GLOBAL_SCREEN[
        "global_grad_ratio_bf16"
    ]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

GLOBAL_SCREEN.to_csv(
    RESULTS / "global_screen_9984.csv",
    index=False,
)

GLOBAL_GRAD_SCREEN_PAIR = tuple(
    GLOBAL_SCREEN.sort_values(
        "global_target_grad_bf16",
        ascending=False,
    )
    .iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

GLOBAL_GRAD_SPEC_SCREEN_PAIR = tuple(
    GLOBAL_SCREEN.sort_values(
        "global_grad_specific_bf16",
        ascending=False,
    )
    .iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

print("Global raw-gradient screen winner:", GLOBAL_GRAD_SCREEN_PAIR)
print("Global gradient-specific screen winner:", GLOBAL_GRAD_SPEC_SCREEN_PAIR)

display(
    GLOBAL_SCREEN.sort_values(
        "global_target_grad_bf16",
        ascending=False,
    ).head(20)
)

# Phase E — Construct the 72-expert atlas panel

The panel has two analytically different components.

### 48 population experts

These are sampled independently of causal, routing, or gradient rank.
The sample guarantees broad layer coverage:

- one uniformly random expert from each of the 39 sparse layers;
- nine additional uniformly random expert positions.

**Only these 48 are used for the primary population correlation estimates.**

### 24 sentinel experts

Sentinels deliberately cover extremes:

- v6 anchors L36/E229, L29/E194, L25/E168;
- top global target-gradient experts;
- top global gradient-specific experts;
- top global routing experts.

They test ranking quality and extreme winners, but are excluded from the
primary population-correlation estimate to avoid selection-induced inflation.

In [ ]:
PANEL_SEED = 2027
POPULATION_N = 48
SENTINEL_N = 24

panel_rng = np.random.default_rng(
    PANEL_SEED
)

all_pairs = [
    (int(layer_idx), int(expert_id))
    for layer_idx in SPARSE_LAYERS
    for expert_id in range(cfg.num_experts)
]

# One random expert per sparse layer.
population_pairs = []

for layer_idx in SPARSE_LAYERS:
    expert_id = int(
        panel_rng.integers(
            0,
            cfg.num_experts,
        )
    )
    population_pairs.append(
        (int(layer_idx), expert_id)
    )

population_set = set(population_pairs)

# Add nine globally random positions.
remaining = [
    p
    for p in all_pairs
    if p not in population_set
]

extra_idx = panel_rng.choice(
    len(remaining),
    size=POPULATION_N - len(population_pairs),
    replace=False,
)

for idx in extra_idx:
    population_pairs.append(
        remaining[int(idx)]
    )

population_set = set(population_pairs)

assert len(population_set) == POPULATION_N

V6_ANCHORS = [
    (36, 229),  # causal
    (29, 194),  # routing
    (25, 168),  # gradient
]

ordered_sentinel_candidates = []

def add_ranked(df, sort_col, n):
    ranked = df.sort_values(
        sort_col,
        ascending=False,
    ).head(n)

    for r in ranked.itertuples(index=False):
        ordered_sentinel_candidates.append(
            (
                int(r.layer),
                int(r.expert),
            )
        )

ordered_sentinel_candidates.extend(
    V6_ANCHORS
)

add_ranked(
    GLOBAL_SCREEN,
    "global_target_grad_bf16",
    12,
)

add_ranked(
    GLOBAL_SCREEN,
    "global_grad_specific_bf16",
    12,
)

add_ranked(
    GLOBAL_SCREEN,
    "selection_supervised_routing_mass",
    12,
)

add_ranked(
    GLOBAL_SCREEN,
    "global_grad_ratio_bf16",
    12,
)

sentinel_pairs = []

for pair in ordered_sentinel_candidates:
    if pair in population_set:
        continue
    if pair in sentinel_pairs:
        continue

    sentinel_pairs.append(pair)

    if len(sentinel_pairs) == SENTINEL_N:
        break

if len(sentinel_pairs) < SENTINEL_N:
    for pair in all_pairs:
        if pair in population_set:
            continue
        if pair in sentinel_pairs:
            continue

        sentinel_pairs.append(pair)

        if len(sentinel_pairs) == SENTINEL_N:
            break

assert len(sentinel_pairs) == SENTINEL_N

panel_rows = []

for pair in population_pairs:
    panel_rows.append({
        "layer": pair[0],
        "expert": pair[1],
        "panel_group": "population_random",
    })

for pair in sentinel_pairs:
    panel_rows.append({
        "layer": pair[0],
        "expert": pair[1],
        "panel_group": "sentinel",
    })

panel_df = pd.DataFrame(
    panel_rows
).drop_duplicates(
    ["layer", "expert"]
).reset_index(drop=True)

if len(panel_df) != POPULATION_N + SENTINEL_N:
    raise RuntimeError(
        f"Panel size mismatch: {len(panel_df)}"
    )

for layer_idx, expert_id in V6_ANCHORS:
    mask = (
        (panel_df["layer"] == layer_idx)
        & (panel_df["expert"] == expert_id)
    )
    panel_df.loc[
        mask,
        "v6_anchor"
    ] = True

panel_df["v6_anchor"] = (
    panel_df["v6_anchor"]
    .fillna(False)
    .astype(bool)
)

panel_df = panel_df.merge(
    GLOBAL_SCREEN,
    on=["layer", "expert"],
    how="left",
)

panel_df.to_csv(
    PANEL_CSV,
    index=False,
)

print(
    "Panel:",
    len(panel_df),
    "=",
    (panel_df["panel_group"] == "population_random").sum(),
    "population +",
    (panel_df["panel_group"] == "sentinel").sum(),
    "sentinels",
)

display(
    panel_df[
        [
            "layer",
            "expert",
            "panel_group",
            "v6_anchor",
            "global_target_grad_rank",
            "global_grad_specific_rank",
            "routing_rank",
        ]
    ].head(30)
)

# Phase F — Exact-baseline FP32 surgical bank

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import types


class SurgicalExpertBank(nn.Module):
    """
    Exact-baseline delta surgery.

    Forward =
        original_frozen_expert_output
        + trainable_selected_expert_output
        - frozen_selected_expert_output

    At initialization:
        trainable_selected == frozen_selected

    therefore:
        delta == 0 exactly

    and the untouched model's original expert kernel remains the base path.
    """

    def __init__(self, selected_pairs):
        super().__init__()

        self.selected_pairs = sorted({
            (int(layer_idx), int(expert_id))
            for layer_idx, expert_id in selected_pairs
        })

        self.params = nn.ParameterDict()
        self.key_map = {}

        self.original_forwards = {}
        self.installed = False

        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            gu_key = f"L{layer_idx}_E{expert_id}_gu"
            down_key = f"L{layer_idx}_E{expert_id}_down"

            # FP32 master parameters.
            self.params[gu_key] = nn.Parameter(
                experts.gate_up_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.params[down_key] = nn.Parameter(
                experts.down_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.key_map[
                (layer_idx, expert_id)
            ] = (
                gu_key,
                down_key,
            )

        self.to("cuda:0")

    @property
    def trainable_parameter_count(self):
        return sum(
            p.numel()
            for p in self.parameters()
        )

    def install(self):
        if self.installed:
            return

        bank = self

        selected_layers = sorted({
            layer_idx
            for layer_idx, _ in self.selected_pairs
        })

        for layer_idx in selected_layers:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            original_forward = experts.forward

            self.original_forwards[
                layer_idx
            ] = original_forward

            selected_ids = sorted({
                expert_id
                for l, expert_id in self.selected_pairs
                if l == layer_idx
            })

            def make_forward(
                idx,
                base_experts,
                original_fn,
                selected_expert_ids,
            ):
                def patched_forward(
                    self_experts,
                    hidden_states,
                    top_k_index,
                    top_k_weights,
                ):
                    # -------------------------------------------------
                    # IMPORTANT:
                    # Preserve Laguna's ORIGINAL execution path.
                    # -------------------------------------------------
                    base_output = original_fn(
                        hidden_states,
                        top_k_index,
                        top_k_weights,
                    )

                    correction = torch.zeros_like(
                        base_output
                    )

                    for expert_id in selected_expert_ids:

                        # top_k_index:
                        # [num_tokens, top_k]
                        token_idx, top_k_pos = torch.where(
                            top_k_index == expert_id
                        )

                        if token_idx.numel() == 0:
                            continue

                        current_state = hidden_states[
                            token_idx
                        ]

                        compute_dtype = (
                            current_state.dtype
                        )

                        gu_key, down_key = (
                            bank.key_map[
                                (idx, expert_id)
                            ]
                        )

                        # ---------------------------------------------
                        # Trainable FP32 master -> current compute dtype
                        # Gradient propagates through .to(dtype).
                        # ---------------------------------------------
                        train_gu = bank.params[
                            gu_key
                        ].to(compute_dtype)

                        train_down = bank.params[
                            down_key
                        ].to(compute_dtype)

                        # ---------------------------------------------
                        # Frozen reference expert.
                        # ---------------------------------------------
                        frozen_gu = (
                            base_experts
                            .gate_up_proj[
                                expert_id
                            ]
                            .detach()
                        )

                        frozen_down = (
                            base_experts
                            .down_proj[
                                expert_id
                            ]
                            .detach()
                        )

                        # ---------------------------------------------
                        # Trainable selected expert.
                        # ---------------------------------------------
                        train_gate, train_up = F.linear(
                            current_state,
                            train_gu,
                        ).chunk(
                            2,
                            dim=-1,
                        )

                        train_hidden = (
                            base_experts.act_fn(
                                train_gate
                            )
                            * train_up
                        )

                        train_hidden = F.linear(
                            train_hidden,
                            train_down,
                        )

                        # ---------------------------------------------
                        # Frozen selected expert using EXACT SAME
                        # manual computation as trainable branch.
                        #
                        # Therefore at initialization:
                        # train_hidden - frozen_hidden == 0.
                        # ---------------------------------------------
                        frozen_gate, frozen_up = F.linear(
                            current_state,
                            frozen_gu,
                        ).chunk(
                            2,
                            dim=-1,
                        )

                        frozen_hidden = (
                            base_experts.act_fn(
                                frozen_gate
                            )
                            * frozen_up
                        )

                        frozen_hidden = F.linear(
                            frozen_hidden,
                            frozen_down,
                        )

                        route_weight = top_k_weights[
                            token_idx,
                            top_k_pos,
                            None,
                        ]

                        train_hidden = (
                            train_hidden
                            * route_weight
                        )

                        frozen_hidden = (
                            frozen_hidden
                            * route_weight
                        )

                        delta = (
                            train_hidden
                            - frozen_hidden
                        ).to(
                            base_output.dtype
                        )

                        # Out-of-place index_add preserves autograd.
                        correction = correction.index_add(
                            0,
                            token_idx,
                            delta,
                        )

                    return (
                        base_output
                        + correction
                    )

                return patched_forward

            experts.forward = types.MethodType(
                make_forward(
                    layer_idx,
                    experts,
                    original_forward,
                    selected_ids,
                ),
                experts,
            )

        self.installed = True

    def restore(self):
        if not self.installed:
            return

        for layer_idx, original_forward in (
            self.original_forwards.items()
        ):
            get_sparse_mlp(
                layer_idx
            ).experts.forward = (
                original_forward
            )

        self.original_forwards.clear()
        self.installed = False

## Verify exact baseline equivalence before any atlas training

In [ ]:
def verify_routed_bank_equivalence(
    pair,
    tol=1e-5,
):
    probe_df = selection_df.head(8)
    batch = build_scoring_batch(
        probe_df
    )

    before = score_batch(batch)

    bank = SurgicalExpertBank([pair])
    bank.install()

    try:
        after = score_batch(batch)
    finally:
        bank.restore()
        del bank
        gc.collect()
        torch.cuda.empty_cache()

    diff = float(
        np.max(np.abs(before - after))
    )

    print(
        "Routed bank equivalence",
        pair,
        "max ΔNLL:",
        diff,
    )

    if diff > tol:
        raise RuntimeError(
            "SurgicalExpertBank changes outputs before training."
        )

equivalence_pairs = {
    (36, 229),
    (29, 194),
    (25, 168),
    GLOBAL_GRAD_SCREEN_PAIR,
    GLOBAL_GRAD_SPEC_SCREEN_PAIR,
    GLOBAL_ROUTING_PAIR,
}

for pair in sorted(equivalence_pairs):
    verify_routed_bank_equivalence(
        pair
    )

print("Surgery equivalence: PASS")

# Phase G — Exact causal + renormalized intervention for all 72 panel experts

In [ ]:
RUN_PANEL_CAUSAL = True

if PANEL_CAUSAL_CSV.exists():
    panel_causal_partial = pd.read_csv(
        PANEL_CAUSAL_CSV
    )
else:
    panel_causal_partial = pd.DataFrame()

completed_causal = set()

if not panel_causal_partial.empty:
    completed_causal = set(
        zip(
            panel_causal_partial["layer"].astype(int),
            panel_causal_partial["expert"].astype(int),
        )
    )

if RUN_PANEL_CAUSAL:
    for r in tqdm(
        list(panel_df.itertuples(index=False)),
        desc="Panel causal interventions",
    ):
        pair = (
            int(r.layer),
            int(r.expert),
        )

        if pair in completed_causal:
            continue

        normal = intervention_score(
            pair[0],
            [pair[1]],
            renormalize=False,
        )

        renorm = intervention_score(
            pair[0],
            [pair[1]],
            renormalize=True,
        )

        boot = bootstrap_specificity(
            normal["per_example_delta"],
            n_boot=5000,
            seed=(
                7000
                + pair[0] * 257
                + pair[1]
            ),
        )

        row = {
            "layer": pair[0],
            "expert": pair[1],
            "target_delta_nll": normal["target_delta_nll"],
            "control_delta_nll": normal["control_delta_nll"],
            "causal_specificity": normal["causal_specificity"],
            "renorm_target_delta_nll": renorm["target_delta_nll"],
            "renorm_control_delta_nll": renorm["control_delta_nll"],
            "renorm_causal_specificity": renorm["causal_specificity"],
            **boot,
        }

        panel_causal_partial = pd.concat(
            [
                panel_causal_partial,
                pd.DataFrame([row]),
            ],
            ignore_index=True,
        )

        panel_causal_partial.to_csv(
            PANEL_CAUSAL_CSV,
            index=False,
        )

panel_causal_df = pd.read_csv(
    PANEL_CAUSAL_CSV
)

if len(panel_causal_df) != len(panel_df):
    raise RuntimeError(
        "Panel causal phase incomplete."
    )

display(
    panel_causal_df.sort_values(
        "causal_specificity",
        ascending=False,
    ).head(20)
)

# Phase H — Precise FP32 target/control gradient geometry for panel experts

For each panel expert v7 measures, using the exact trainable delta path:

- target gradient L2;
- control gradient L2;
- target/control gradient dot product;
- target/control gradient cosine;
- gradient-specific score.

The cosine helps distinguish **broad writeability** from gradient directions
that differ between target and control behavior.

In [ ]:
PRECISE_GRAD_CASES = 12
RUN_PRECISE_PANEL_GRADIENTS = True

def _configure_grad_checkpointing():
    if hasattr(
        model,
        "enable_input_require_grads",
    ):
        model.enable_input_require_grads()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )
    except TypeError:
        model.gradient_checkpointing_enable()

def _disable_grad_checkpointing():
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass

    if hasattr(
        model,
        "disable_input_require_grads",
    ):
        model.disable_input_require_grads()

def _backprop_bank_cases(
    bank,
    cases,
    max_cases,
):
    used = min(
        int(max_cases),
        len(cases),
    )

    for p in bank.parameters():
        p.grad = None

    for case in cases[:used]:
        with torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ):
            out = model(
                input_ids=case["input_ids"],
                attention_mask=case["attention_mask"],
                use_cache=False,
                logits_to_keep=case["pred_positions"],
                return_dict=True,
            )

            logits = out.logits.float()

            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                case["targets"].reshape(-1),
            ) / used

        loss.backward()

        del out, logits, loss

    grads = []

    for p in bank.parameters():
        if p.grad is None:
            grads.append(
                torch.zeros_like(
                    p,
                    device="cpu",
                    dtype=torch.float32,
                )
            )
        else:
            grads.append(
                p.grad.detach().float().cpu().clone()
            )

    return grads

def precise_gradient_geometry(
    pair,
    target_cases,
    control_cases,
    max_cases=12,
):
    pair = tuple(map(int, pair))

    bank = SurgicalExpertBank([pair])
    bank.install()

    for p in model.parameters():
        p.requires_grad_(False)

    _configure_grad_checkpointing()

    model.train()
    bank.train()

    try:
        target_grads = _backprop_bank_cases(
            bank,
            target_cases,
            max_cases,
        )

        control_grads = _backprop_bank_cases(
            bank,
            control_cases,
            max_cases,
        )

        target_sq = 0.0
        control_sq = 0.0
        dot = 0.0

        for gt, gc_ in zip(
            target_grads,
            control_grads,
        ):
            target_sq += float(
                torch.sum(gt * gt).item()
            )
            control_sq += float(
                torch.sum(gc_ * gc_).item()
            )
            dot += float(
                torch.sum(gt * gc_).item()
            )

        target_l2 = float(
            np.sqrt(target_sq)
        )
        control_l2 = float(
            np.sqrt(control_sq)
        )

        cosine = float(
            dot
            / max(
                target_l2 * control_l2,
                1e-12,
            )
        )

        return {
            "precise_target_grad_l2": target_l2,
            "precise_control_grad_l2": control_l2,
            "precise_grad_dot": dot,
            "precise_grad_cosine": cosine,
            "precise_grad_specific": (
                target_l2
                - CONTROL_PENALTY * control_l2
            ),
            "precise_grad_ratio": (
                target_l2
                / (control_l2 + 1e-12)
            ),
        }

    finally:
        bank.restore()
        _disable_grad_checkpointing()
        model.eval()

        del bank
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
if PANEL_GRAD_CSV.exists():
    panel_grad_partial = pd.read_csv(
        PANEL_GRAD_CSV
    )
else:
    panel_grad_partial = pd.DataFrame()

completed_grad = set()

if not panel_grad_partial.empty:
    completed_grad = set(
        zip(
            panel_grad_partial["layer"].astype(int),
            panel_grad_partial["expert"].astype(int),
        )
    )

if RUN_PRECISE_PANEL_GRADIENTS:
    for r in tqdm(
        list(panel_df.itertuples(index=False)),
        desc="Panel precise gradients",
    ):
        pair = (
            int(r.layer),
            int(r.expert),
        )

        if pair in completed_grad:
            continue

        metrics = precise_gradient_geometry(
            pair,
            SELECTION_TARGET_CASES,
            SELECTION_CONTROL_CASES,
            max_cases=PRECISE_GRAD_CASES,
        )

        row = {
            "layer": pair[0],
            "expert": pair[1],
            **metrics,
        }

        panel_grad_partial = pd.concat(
            [
                panel_grad_partial,
                pd.DataFrame([row]),
            ],
            ignore_index=True,
        )

        panel_grad_partial.to_csv(
            PANEL_GRAD_CSV,
            index=False,
        )

panel_grad_df = pd.read_csv(
    PANEL_GRAD_CSV
)

if len(panel_grad_df) != len(panel_df):
    raise RuntimeError(
        "Panel precise-gradient phase incomplete."
    )

display(
    panel_grad_df.sort_values(
        "precise_target_grad_l2",
        ascending=False,
    ).head(20)
)

# Phase I — Validation and final-test adaptation metrics

In [ ]:
@torch.inference_mode()
def score_batch_detailed(batch):
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    nll = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    preds = logits.argmax(dim=-1)

    correct = (
        preds.eq(targets)
        & valid
    )

    token_acc = (
        correct.sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    seq_exact = (
        (correct | ~valid)
        .all(dim=-1)
        .float()
    )

    result = {
        "nll": nll.cpu().numpy(),
        "token_acc": token_acc.cpu().numpy(),
        "seq_exact": seq_exact.cpu().numpy(),
    }

    del out, logits, losses, preds

    return result

VALIDATION_BASE_DETAIL = score_batch_detailed(
    VALIDATION_BATCH
)

FINAL_TEST_BASE_DETAIL = score_batch_detailed(
    FINAL_TEST_BATCH
)

def evaluate_batch_against_base(
    batch,
    df,
    base_detail,
):
    d = score_batch_detailed(batch)

    target_mask = (
        df["kind"].values == "target"
    )
    control_mask = (
        df["kind"].values == "control"
    )

    target_nll = float(
        d["nll"][target_mask].mean()
    )
    control_nll = float(
        d["nll"][control_mask].mean()
    )

    base_target_nll = float(
        base_detail["nll"][
            target_mask
        ].mean()
    )
    base_control_nll = float(
        base_detail["nll"][
            control_mask
        ].mean()
    )

    target_improvement = (
        base_target_nll - target_nll
    )

    control_improvement = (
        base_control_nll - control_nll
    )

    control_damage = (
        -control_improvement
    )

    utility_score = (
        target_improvement
        - CONTROL_PENALTY
        * max(control_damage, 0.0)
    )

    specific_gain = (
        target_improvement
        - control_improvement
    )

    return {
        "target_nll": target_nll,
        "control_nll": control_nll,
        "target_improvement": float(target_improvement),
        "control_improvement": float(control_improvement),
        "control_damage": float(control_damage),
        "utility_score": float(utility_score),
        "specific_gain": float(specific_gain),
        "target_token_acc": float(
            d["token_acc"][target_mask].mean()
        ),
        "control_token_acc": float(
            d["token_acc"][control_mask].mean()
        ),
        "target_seq_exact": float(
            d["seq_exact"][target_mask].mean()
        ),
        "control_seq_exact": float(
            d["seq_exact"][control_mask].mean()
        ),
        "per_example_nll": d["nll"],
    }

# Phase J — Matched routed-expert training engine

In [ ]:
def capture_routed_base_guard(pairs):
    guard = {}

    for layer_idx, expert_id in pairs:
        experts = get_sparse_mlp(
            layer_idx
        ).experts

        guard[(layer_idx, expert_id)] = (
            experts.gate_up_proj[
                expert_id
            ].detach().cpu().clone(),
            experts.down_proj[
                expert_id
            ].detach().cpu().clone(),
        )

    return guard

def assert_routed_base_unchanged(guard):
    for (
        layer_idx,
        expert_id,
    ), (gu0, down0) in guard.items():
        experts = get_sparse_mlp(
            layer_idx
        ).experts

        gu1 = (
            experts.gate_up_proj[
                expert_id
            ].detach().cpu()
        )

        down1 = (
            experts.down_proj[
                expert_id
            ].detach().cpu()
        )

        if not torch.equal(gu0, gu1):
            raise RuntimeError(
                f"Frozen base gate_up changed: "
                f"L{layer_idx}/E{expert_id}"
            )

        if not torch.equal(down0, down1):
            raise RuntimeError(
                f"Frozen base down changed: "
                f"L{layer_idx}/E{expert_id}"
            )

def state_l2(state):
    total = 0.0

    for v in state.values():
        x = v.detach().float()
        total += float(
            torch.sum(x * x).item()
        )

    return float(np.sqrt(total))

def state_delta_l2(before, after):
    total = 0.0

    for k in before:
        d = (
            after[k].detach().float()
            - before[k].detach().float()
        )
        total += float(
            torch.sum(d * d).item()
        )

    return float(np.sqrt(total))

In [ ]:
def train_routed_experts(
    selector,
    pairs,
    order_seed,
    lr,
    epochs,
    max_updates,
    grad_accum,
    eval_batch,
    eval_df,
    eval_base_detail,
):
    pairs = sorted({
        tuple(map(int, pair))
        for pair in pairs
    })

    bank = SurgicalExpertBank(
        pairs
    )

    expected_params = (
        len(pairs) * params_per_expert
    )

    if (
        bank.trainable_parameter_count
        != expected_params
    ):
        raise RuntimeError(
            "Trainable parameter budget mismatch."
        )

    guard = capture_routed_base_guard(
        pairs
    )

    bank.install()

    for p in model.parameters():
        p.requires_grad_(False)

    _configure_grad_checkpointing()

    model.train()
    bank.train()

    optimizer = torch.optim.AdamW(
        bank.parameters(),
        lr=float(lr),
        betas=(0.9, 0.95),
        weight_decay=0.01,
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    initial_state = {
        k: v.detach().cpu().clone()
        for k, v in bank.state_dict().items()
    }

    rng = np.random.default_rng(
        int(order_seed)
    )

    history = []
    raw_step = 0
    update_step = 0

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        for epoch in range(int(epochs)):
            order = rng.permutation(
                len(TRAIN_CASES)
            ).tolist()

            for position, case_idx in enumerate(
                order
            ):
                case = TRAIN_CASES[
                    int(case_idx)
                ]

                raw_step += 1

                with torch.autocast(
                    "cuda",
                    dtype=torch.bfloat16,
                ):
                    out = model(
                        input_ids=case["input_ids"],
                        attention_mask=case["attention_mask"],
                        use_cache=False,
                        logits_to_keep=case["pred_positions"],
                        return_dict=True,
                    )

                    logits = out.logits.float()

                    loss = F.cross_entropy(
                        logits.reshape(
                            -1,
                            logits.shape[-1],
                        ),
                        case["targets"].reshape(-1),
                    )

                    scaled = (
                        loss / int(grad_accum)
                    )

                scaled.backward()

                is_final_available = (
                    epoch == int(epochs) - 1
                    and position == len(order) - 1
                )

                should_step = (
                    raw_step % int(grad_accum) == 0
                    or is_final_available
                )

                if should_step:
                    grad_norm = (
                        torch.nn.utils.clip_grad_norm_(
                            bank.parameters(),
                            1.0,
                        )
                    )

                    optimizer.step()
                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    update_step += 1

                    history.append({
                        "selector": selector,
                        "budget_k": len(pairs),
                        "order_seed": int(order_seed),
                        "lr": float(lr),
                        "update_step": int(update_step),
                        "raw_step": int(raw_step),
                        "loss": float(loss.detach().item()),
                        "grad_norm": float(grad_norm),
                    })

                del out, logits, loss, scaled

                if (
                    update_step
                    >= int(max_updates)
                ):
                    break

            if (
                update_step
                >= int(max_updates)
            ):
                break

        model.eval()
        bank.eval()

        metrics = evaluate_batch_against_base(
            eval_batch,
            eval_df,
            eval_base_detail,
        )

        final_state = {
            k: v.detach().cpu().clone()
            for k, v in bank.state_dict().items()
        }

        delta_l2 = state_delta_l2(
            initial_state,
            final_state,
        )

        base_l2 = state_l2(
            initial_state
        )

        result = {
            "selector": selector,
            "pairs": pairs,
            "budget_k": len(pairs),
            "order_seed": int(order_seed),
            "lr": float(lr),
            "epochs": int(epochs),
            "max_updates": int(max_updates),
            "updates": int(update_step),
            "trainable_params": int(
                bank.trainable_parameter_count
            ),
            "mean_grad_norm": float(
                np.mean([
                    h["grad_norm"]
                    for h in history
                ])
            ) if history else 0.0,
            "parameter_delta_l2": float(delta_l2),
            "relative_parameter_delta": float(
                delta_l2
                / max(base_l2, 1e-12)
            ),
            "peak_gpu_gib": float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
            **{
                k: v
                for k, v in metrics.items()
                if k != "per_example_nll"
            },
            "per_example_nll": metrics["per_example_nll"],
            "history": history,
        }

        return result

    finally:
        bank.restore()
        _disable_grad_checkpointing()
        model.eval()

        assert_routed_base_unchanged(
            guard
        )

        del optimizer
        del bank
        del guard

        gc.collect()
        torch.cuda.empty_cache()

# Phase K — One-epoch adaptation atlas on validation

Every panel expert gets the same:

- one routed expert = 3,145,728 trainable parameters;
- same 50 target training examples;
- same example order;
- same LR;
- same gradient accumulation;
- maximum 7 optimizer updates;
- same 25-target + 25-control validation evaluation.

This phase measures \(A_i\) cheaply enough to cover 72 experts.

The untouched final-test split is **not used here**.

In [ ]:
RUN_PANEL_ADAPTATION = True

ATLAS_LR = 1e-5
ATLAS_EPOCHS = 1
ATLAS_MAX_UPDATES = 7
ATLAS_GRAD_ACCUM = 8
ATLAS_ORDER_SEED = 101

ATLAS_NLL_DIR = RESULTS / "atlas_validation_nll"
ATLAS_NLL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if PANEL_ADAPT_CSV.exists():
    atlas_adapt_partial = pd.read_csv(
        PANEL_ADAPT_CSV
    )
else:
    atlas_adapt_partial = pd.DataFrame()

completed_adapt = set()

if not atlas_adapt_partial.empty:
    completed_adapt = set(
        zip(
            atlas_adapt_partial["layer"].astype(int),
            atlas_adapt_partial["expert"].astype(int),
        )
    )

if RUN_PANEL_ADAPTATION:
    for r in tqdm(
        list(panel_df.itertuples(index=False)),
        desc="72-expert adaptation atlas",
    ):
        pair = (
            int(r.layer),
            int(r.expert),
        )

        if pair in completed_adapt:
            continue

        result = train_routed_experts(
            selector=f"L{pair[0]}_E{pair[1]}",
            pairs=[pair],
            order_seed=ATLAS_ORDER_SEED,
            lr=ATLAS_LR,
            epochs=ATLAS_EPOCHS,
            max_updates=ATLAS_MAX_UPDATES,
            grad_accum=ATLAS_GRAD_ACCUM,
            eval_batch=VALIDATION_BATCH,
            eval_df=validation_df,
            eval_base_detail=VALIDATION_BASE_DETAIL,
        )

        row = {
            "layer": pair[0],
            "expert": pair[1],
            "atlas_order_seed": ATLAS_ORDER_SEED,
            "atlas_lr": ATLAS_LR,
            "atlas_updates": result["updates"],
            "trainable_params": result["trainable_params"],
            "mean_train_grad_norm": result["mean_grad_norm"],
            "parameter_delta_l2": result["parameter_delta_l2"],
            "relative_parameter_delta": result["relative_parameter_delta"],
            "peak_gpu_gib": result["peak_gpu_gib"],
            "validation_target_nll": result["target_nll"],
            "validation_control_nll": result["control_nll"],
            "validation_target_improvement": result["target_improvement"],
            "validation_control_improvement": result["control_improvement"],
            "validation_utility_score": result["utility_score"],
            "validation_specific_gain": result["specific_gain"],
            "validation_target_token_acc": result["target_token_acc"],
            "validation_control_token_acc": result["control_token_acc"],
            "validation_target_seq_exact": result["target_seq_exact"],
            "validation_control_seq_exact": result["control_seq_exact"],
        }

        atlas_adapt_partial = pd.concat(
            [
                atlas_adapt_partial,
                pd.DataFrame([row]),
            ],
            ignore_index=True,
        )

        atlas_adapt_partial.to_csv(
            PANEL_ADAPT_CSV,
            index=False,
        )

        np.save(
            ATLAS_NLL_DIR
            / f"L{pair[0]}_E{pair[1]}.npy",
            result["per_example_nll"],
        )

        print(
            pair,
            "target=",
            f"{result['target_improvement']:+.4f}",
            "specific=",
            f"{result['specific_gain']:+.4f}",
        )

atlas_adapt_df = pd.read_csv(
    PANEL_ADAPT_CSV
)

if len(atlas_adapt_df) != len(panel_df):
    raise RuntimeError(
        "Panel adaptation phase incomplete."
    )

display(
    atlas_adapt_df.sort_values(
        "validation_target_improvement",
        ascending=False,
    ).head(20)
)

# Phase L — Build the complete C/R/G/A atlas table

In [ ]:
ATLAS = (
    panel_df
    .merge(
        panel_causal_df,
        on=["layer", "expert"],
        how="inner",
    )
    .merge(
        panel_grad_df,
        on=["layer", "expert"],
        how="inner",
    )
    .merge(
        atlas_adapt_df,
        on=["layer", "expert"],
        how="inner",
    )
)

ATLAS.to_csv(
    RESULTS / "complete_crga_atlas.csv",
    index=False,
)

print("Complete atlas rows:", len(ATLAS))

display(
    ATLAS.sort_values(
        "validation_target_improvement",
        ascending=False,
    )[
        [
            "layer",
            "expert",
            "panel_group",
            "causal_specificity",
            "selection_supervised_routing_mass",
            "precise_target_grad_l2",
            "precise_grad_specific",
            "precise_grad_cosine",
            "validation_target_improvement",
            "validation_specific_gain",
        ]
    ].head(25)
)

## Primary population correlations

**Primary analysis = 48 population-sampled experts only.**

Sentinels are excluded here because intentionally selecting score extremes
would artificially change correlation estimates.

For each predictor, v7 reports:

- Spearman \(\rho\);
- 95% bootstrap interval;
- Kendall \(\tau\);
- target-improvement prediction;
- target-specific-gain prediction.

In [ ]:
from scipy.stats import (
    spearmanr,
    kendalltau,
    rankdata,
    pearsonr,
)

POP = ATLAS[
    ATLAS["panel_group"]
    == "population_random"
].reset_index(drop=True)

if len(POP) != POPULATION_N:
    raise RuntimeError(
        f"Population panel mismatch: {len(POP)}"
    )

PREDICTORS = {
    "causal_specificity": "causal_specificity",
    "renorm_causal_specificity": "renorm_causal_specificity",
    "routing_mass": "selection_supervised_routing_mass",
    "routing_rate": "selection_supervised_selected_rate",
    "global_target_gradient": "global_target_grad_bf16",
    "global_gradient_specific": "global_grad_specific_bf16",
    "precise_target_gradient": "precise_target_grad_l2",
    "precise_control_gradient": "precise_control_grad_l2",
    "precise_gradient_specific": "precise_grad_specific",
    "precise_gradient_ratio": "precise_grad_ratio",
    "target_control_gradient_cosine": "precise_grad_cosine",
    "initial_to_train_gradient": "mean_train_grad_norm",
}

OUTCOMES = {
    "target_improvement": "validation_target_improvement",
    "specific_gain": "validation_specific_gain",
}

def bootstrap_spearman(
    x,
    y,
    n_boot=5000,
    seed=1234,
):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    rng = np.random.default_rng(seed)

    vals = []

    for _ in range(n_boot):
        idx = rng.integers(
            0,
            len(x),
            size=len(x),
        )

        xb = x[idx]
        yb = y[idx]

        if (
            np.nanstd(xb) == 0
            or np.nanstd(yb) == 0
        ):
            continue

        rho = spearmanr(
            xb,
            yb,
            nan_policy="omit",
        ).statistic

        if np.isfinite(rho):
            vals.append(float(rho))

    vals = np.asarray(vals)

    if len(vals) == 0:
        return np.nan, np.nan

    return (
        float(np.quantile(vals, 0.025)),
        float(np.quantile(vals, 0.975)),
    )

corr_rows = []

for pred_name, pred_col in PREDICTORS.items():
    for out_name, out_col in OUTCOMES.items():
        x = POP[pred_col].to_numpy(
            dtype=np.float64
        )
        y = POP[out_col].to_numpy(
            dtype=np.float64
        )

        valid = (
            np.isfinite(x)
            & np.isfinite(y)
        )

        x = x[valid]
        y = y[valid]

        sp = spearmanr(
            x,
            y,
            nan_policy="omit",
        )

        kt = kendalltau(
            x,
            y,
            nan_policy="omit",
        )

        ci_low, ci_high = (
            bootstrap_spearman(
                x,
                y,
                n_boot=5000,
                seed=(
                    17000
                    + len(corr_rows)
                ),
            )
        )

        corr_rows.append({
            "predictor": pred_name,
            "outcome": out_name,
            "n": len(x),
            "spearman_rho": float(
                sp.statistic
            ),
            "spearman_p": float(
                sp.pvalue
            ),
            "spearman_ci_low": ci_low,
            "spearman_ci_high": ci_high,
            "kendall_tau": float(
                kt.statistic
            ),
            "kendall_p": float(
                kt.pvalue
            ),
        })

correlation_df = pd.DataFrame(
    corr_rows
).sort_values(
    [
        "outcome",
        "spearman_rho",
    ],
    ascending=[
        True,
        False,
    ],
)

correlation_df.to_csv(
    CORRELATION_CSV,
    index=False,
)

display(correlation_df)

## Partial-rank diagnostics controlling for routing and depth

In [ ]:
from sklearn.linear_model import LinearRegression

def rank01(x):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    r = rankdata(
        x,
        method="average",
    )

    if len(r) <= 1:
        return np.zeros_like(r)

    return (
        (r - 1)
        / (len(r) - 1)
    )

def partial_rank_corr(
    df,
    x_col,
    y_col,
    controls,
):
    cols = [
        x_col,
        y_col,
        *controls,
    ]

    sub = df[
        cols
    ].replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna()

    x = rank01(
        sub[x_col].values
    ).reshape(-1, 1)

    y = rank01(
        sub[y_col].values
    ).reshape(-1, 1)

    Z = np.column_stack([
        rank01(
            sub[c].values
        )
        for c in controls
    ])

    x_res = (
        x.ravel()
        - LinearRegression()
        .fit(Z, x.ravel())
        .predict(Z)
    )

    y_res = (
        y.ravel()
        - LinearRegression()
        .fit(Z, y.ravel())
        .predict(Z)
    )

    return float(
        pearsonr(
            x_res,
            y_res,
        ).statistic
    )

partial_rows = []

for pred in [
    "causal_specificity",
    "global_target_grad_bf16",
    "precise_target_grad_l2",
    "precise_grad_specific",
]:
    for outcome in [
        "validation_target_improvement",
        "validation_specific_gain",
    ]:
        controls = [
            "selection_supervised_routing_mass",
            "layer",
        ]

        rho = partial_rank_corr(
            POP,
            pred,
            outcome,
            controls,
        )

        partial_rows.append({
            "predictor": pred,
            "outcome": outcome,
            "controls": "routing_mass + layer_depth",
            "partial_rank_corr": rho,
        })

partial_df = pd.DataFrame(
    partial_rows
)

display(partial_df)

partial_df.to_csv(
    RESULTS / "partial_rank_correlations.csv",
    index=False,
)

# Phase M — Publication-style diagnostic plots

In [ ]:
import matplotlib.pyplot as plt

def scatter_with_labels(
    df,
    x_col,
    y_col,
    title,
    xlabel,
    ylabel,
    filename,
):
    fig, ax = plt.subplots(
        figsize=(7.5, 5.5)
    )

    pop = df[
        df["panel_group"]
        == "population_random"
    ]

    sent = df[
        df["panel_group"]
        == "sentinel"
    ]

    ax.scatter(
        pop[x_col],
        pop[y_col],
        s=38,
        alpha=0.75,
        label="population sample",
    )

    ax.scatter(
        sent[x_col],
        sent[y_col],
        s=58,
        marker="x",
        label="sentinels",
    )

    for layer_idx, expert_id in V6_ANCHORS:
        row = df[
            (df["layer"] == layer_idx)
            & (df["expert"] == expert_id)
        ]

        if len(row) == 1:
            x = float(row.iloc[0][x_col])
            y = float(row.iloc[0][y_col])

            ax.annotate(
                f"L{layer_idx}/E{expert_id}",
                (x, y),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(
        RESULTS / filename,
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()

scatter_with_labels(
    ATLAS,
    "causal_specificity",
    "validation_target_improvement",
    "Causal necessity vs later adaptation",
    "Causal specificity C(E)",
    "Validation target improvement A(E)",
    "causal_vs_adaptation.png",
)

scatter_with_labels(
    ATLAS,
    "selection_supervised_routing_mass",
    "validation_target_improvement",
    "Routing access vs later adaptation",
    "Supervised routing mass R(E)",
    "Validation target improvement A(E)",
    "routing_vs_adaptation.png",
)

scatter_with_labels(
    ATLAS,
    "precise_target_grad_l2",
    "validation_target_improvement",
    "Gradient accessibility vs later adaptation",
    "Precise target gradient L2 G(E)",
    "Validation target improvement A(E)",
    "gradient_vs_adaptation.png",
)

scatter_with_labels(
    ATLAS,
    "precise_grad_specific",
    "validation_specific_gain",
    "Gradient specificity vs target-specific adaptation",
    "Target − 0.75×control gradient score",
    "Validation target-specific gain",
    "gradient_specific_vs_specific_gain.png",
)

## 3D C/R/G expert-space projection

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(
    figsize=(8, 6.5)
)

ax = fig.add_subplot(
    111,
    projection="3d",
)

pop = ATLAS[
    ATLAS["panel_group"]
    == "population_random"
]

sent = ATLAS[
    ATLAS["panel_group"]
    == "sentinel"
]

ax.scatter(
    pop["causal_specificity"],
    pop["selection_supervised_routing_mass"],
    pop["precise_target_grad_l2"],
    s=30,
    alpha=0.7,
    label="population sample",
)

ax.scatter(
    sent["causal_specificity"],
    sent["selection_supervised_routing_mass"],
    sent["precise_target_grad_l2"],
    s=55,
    marker="x",
    label="sentinels",
)

ax.set_xlabel("Causal necessity C")
ax.set_ylabel("Routing access R")
ax.set_zlabel("Gradient accessibility G")
ax.set_title("Laguna XS.2 expert C/R/G landscape")
ax.legend()

fig.tight_layout()
fig.savefig(
    RESULTS / "crg_3d_landscape.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

# Phase N — Choose selectors without touching final test

Selection rules:

- **causal:** highest exact causal specificity in the 72-expert panel;
- **routing:** global highest supervised routing mass across all 9,984;
- **raw gradient:** precise FP32 winner among the globally top-12 BF16
  target-gradient screen candidates represented in the panel;
- **gradient-specific:** precise FP32 winner among globally top-12 BF16
  gradient-specific screen candidates represented in the panel;
- **validation write oracle:** highest validation target-specific gain in the
  72-expert adaptation atlas.

The validation oracle is explicitly labeled as such. It is not a zero-shot
selector; it asks how much headroom remains after directly observing
validation plasticity.

In [ ]:
PANEL_CAUSAL_PAIR = tuple(
    ATLAS.sort_values(
        "causal_specificity",
        ascending=False,
    )
    .iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

raw_top_pairs = set(
    tuple(map(int, x))
    for x in (
        GLOBAL_SCREEN.sort_values(
            "global_target_grad_bf16",
            ascending=False,
        )
        .head(12)[["layer", "expert"]]
        .to_numpy()
        .tolist()
    )
)

spec_top_pairs = set(
    tuple(map(int, x))
    for x in (
        GLOBAL_SCREEN.sort_values(
            "global_grad_specific_bf16",
            ascending=False,
        )
        .head(12)[["layer", "expert"]]
        .to_numpy()
        .tolist()
    )
)

raw_candidates = ATLAS[
    ATLAS.apply(
        lambda r: (
            int(r["layer"]),
            int(r["expert"]),
        ) in raw_top_pairs,
        axis=1,
    )
]

spec_candidates = ATLAS[
    ATLAS.apply(
        lambda r: (
            int(r["layer"]),
            int(r["expert"]),
        ) in spec_top_pairs,
        axis=1,
    )
]

if raw_candidates.empty:
    raise RuntimeError(
        "No top global-gradient candidate survived into panel."
    )

if spec_candidates.empty:
    raise RuntimeError(
        "No top gradient-specific candidate survived into panel."
    )

GLOBAL_GRAD_PAIR = tuple(
    raw_candidates.sort_values(
        "precise_target_grad_l2",
        ascending=False,
    )
    .iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

GLOBAL_GRAD_SPEC_PAIR = tuple(
    spec_candidates.sort_values(
        "precise_grad_specific",
        ascending=False,
    )
    .iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

VALIDATION_WRITE_PAIR = tuple(
    ATLAS.sort_values(
        "validation_specific_gain",
        ascending=False,
    )
    .iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

FINAL_SELECTORS = {
    "panel_causal": PANEL_CAUSAL_PAIR,
    "global_routing": GLOBAL_ROUTING_PAIR,
    "global_gradient": GLOBAL_GRAD_PAIR,
    "global_gradient_specific": GLOBAL_GRAD_SPEC_PAIR,
    "validation_write_oracle": VALIDATION_WRITE_PAIR,
}

print("Final selectors:")
for name, pair in FINAL_SELECTORS.items():
    print(f"  {name:26s}", pair)

# Phase O — Fresh full-budget confirmation on untouched final test

In [ ]:
RUN_FINAL_CONFIRMATION = True

CONFIRM_LR = 1e-5
CONFIRM_EPOCHS = 3
CONFIRM_MAX_UPDATES = 50
CONFIRM_GRAD_ACCUM = 8
CONFIRM_ORDER_SEEDS = [11, 23, 47]

if CONFIRM_CSV.exists():
    confirm_partial = pd.read_csv(
        CONFIRM_CSV
    )
else:
    confirm_partial = pd.DataFrame()

completed_confirm = set()

if not confirm_partial.empty:
    completed_confirm = set(
        zip(
            confirm_partial["selector"].astype(str),
            confirm_partial["order_seed"].astype(int),
        )
    )

if RUN_FINAL_CONFIRMATION:
    for selector, pair in FINAL_SELECTORS.items():
        for seed in CONFIRM_ORDER_SEEDS:
            key = (
                str(selector),
                int(seed),
            )

            if key in completed_confirm:
                continue

            print(
                "\nFINAL TEST:",
                selector,
                pair,
                "seed",
                seed,
            )

            result = train_routed_experts(
                selector=selector,
                pairs=[pair],
                order_seed=seed,
                lr=CONFIRM_LR,
                epochs=CONFIRM_EPOCHS,
                max_updates=CONFIRM_MAX_UPDATES,
                grad_accum=CONFIRM_GRAD_ACCUM,
                eval_batch=FINAL_TEST_BATCH,
                eval_df=final_test_df,
                eval_base_detail=FINAL_TEST_BASE_DETAIL,
            )

            row = {
                "selector": selector,
                "layer": pair[0],
                "expert": pair[1],
                "order_seed": seed,
                "lr": CONFIRM_LR,
                "updates": result["updates"],
                "trainable_params": result["trainable_params"],
                "mean_train_grad_norm": result["mean_grad_norm"],
                "relative_parameter_delta": result["relative_parameter_delta"],
                "final_target_improvement": result["target_improvement"],
                "final_control_improvement": result["control_improvement"],
                "final_utility_score": result["utility_score"],
                "final_specific_gain": result["specific_gain"],
                "final_target_token_acc": result["target_token_acc"],
                "final_control_token_acc": result["control_token_acc"],
                "final_target_seq_exact": result["target_seq_exact"],
                "final_control_seq_exact": result["control_seq_exact"],
                "peak_gpu_gib": result["peak_gpu_gib"],
            }

            confirm_partial = pd.concat(
                [
                    confirm_partial,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            confirm_partial.to_csv(
                CONFIRM_CSV,
                index=False,
            )

final_confirmation_df = pd.read_csv(
    CONFIRM_CSV
)

expected_confirm = (
    len(FINAL_SELECTORS)
    * len(CONFIRM_ORDER_SEEDS)
)

if len(final_confirmation_df) != expected_confirm:
    raise RuntimeError(
        f"Final confirmation incomplete: "
        f"{len(final_confirmation_df)} / {expected_confirm}"
    )

display(
    final_confirmation_df.sort_values(
        "final_target_improvement",
        ascending=False,
    )
)

## Aggregate final-test confirmation

In [ ]:
final_summary = (
    final_confirmation_df
    .groupby(
        [
            "selector",
            "layer",
            "expert",
        ]
    )
    .agg(
        n_runs=("order_seed", "count"),
        target_improvement_mean=(
            "final_target_improvement",
            "mean",
        ),
        target_improvement_std=(
            "final_target_improvement",
            "std",
        ),
        control_improvement_mean=(
            "final_control_improvement",
            "mean",
        ),
        specific_gain_mean=(
            "final_specific_gain",
            "mean",
        ),
        specific_gain_std=(
            "final_specific_gain",
            "std",
        ),
        target_token_acc_mean=(
            "final_target_token_acc",
            "mean",
        ),
        target_seq_exact_mean=(
            "final_target_seq_exact",
            "mean",
        ),
        mean_grad_norm=(
            "mean_train_grad_norm",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "target_improvement_mean",
        ascending=False,
    )
)

display(final_summary)

final_summary.to_csv(
    RESULTS / "final_confirmation_summary.csv",
    index=False,
)

# Phase P — K={1,2,4,8} parameter-budget curve

This phase compares global gradient and routing selection as the number of
trainable routed experts grows.

It uses selection-derived rankings only and evaluates once on final test with
a fixed training-order seed.

This is a parameter-efficiency curve, not the primary multi-seed inferential
comparison.

In [ ]:
RUN_BUDGET_CURVE = True
BUDGET_KS = [1, 2, 4, 8]
BUDGET_SEED = 11

gradient_order = [
    (
        int(r.layer),
        int(r.expert),
    )
    for r in GLOBAL_SCREEN.sort_values(
        "global_target_grad_bf16",
        ascending=False,
    ).itertuples(index=False)
]

gradient_specific_order = [
    (
        int(r.layer),
        int(r.expert),
    )
    for r in GLOBAL_SCREEN.sort_values(
        "global_grad_specific_bf16",
        ascending=False,
    ).itertuples(index=False)
]

routing_order = [
    (
        int(r.layer),
        int(r.expert),
    )
    for r in GLOBAL_SCREEN.sort_values(
        "selection_supervised_routing_mass",
        ascending=False,
    ).itertuples(index=False)
]

BUDGET_METHODS = {
    "gradient": gradient_order,
    "gradient_specific": gradient_specific_order,
    "routing": routing_order,
}

if BUDGET_CSV.exists():
    budget_partial = pd.read_csv(
        BUDGET_CSV
    )
else:
    budget_partial = pd.DataFrame()

completed_budget = set()

if not budget_partial.empty:
    completed_budget = set(
        zip(
            budget_partial["method"].astype(str),
            budget_partial["k"].astype(int),
        )
    )

if RUN_BUDGET_CURVE:
    for method, ordered_pairs in BUDGET_METHODS.items():
        for k in BUDGET_KS:
            key = (
                str(method),
                int(k),
            )

            if key in completed_budget:
                continue

            pairs = ordered_pairs[:k]

            result = train_routed_experts(
                selector=f"{method}_K{k}",
                pairs=pairs,
                order_seed=BUDGET_SEED,
                lr=CONFIRM_LR,
                epochs=CONFIRM_EPOCHS,
                max_updates=CONFIRM_MAX_UPDATES,
                grad_accum=CONFIRM_GRAD_ACCUM,
                eval_batch=FINAL_TEST_BATCH,
                eval_df=final_test_df,
                eval_base_detail=FINAL_TEST_BASE_DETAIL,
            )

            row = {
                "method": method,
                "k": int(k),
                "pairs": json.dumps(
                    [list(p) for p in pairs]
                ),
                "trainable_params": result["trainable_params"],
                "updates": result["updates"],
                "target_improvement": result["target_improvement"],
                "control_improvement": result["control_improvement"],
                "utility_score": result["utility_score"],
                "specific_gain": result["specific_gain"],
                "target_token_acc": result["target_token_acc"],
                "target_seq_exact": result["target_seq_exact"],
                "mean_train_grad_norm": result["mean_grad_norm"],
            }

            budget_partial = pd.concat(
                [
                    budget_partial,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            budget_partial.to_csv(
                BUDGET_CSV,
                index=False,
            )

budget_df = pd.read_csv(
    BUDGET_CSV
)

display(
    budget_df.sort_values(
        ["k", "target_improvement"],
        ascending=[True, False],
    )
)

## Budget-curve plots

In [ ]:
fig, ax = plt.subplots(
    figsize=(7.5, 5.2)
)

for method, g in budget_df.groupby(
    "method"
):
    g = g.sort_values("k")

    ax.plot(
        g["k"],
        g["target_improvement"],
        marker="o",
        label=method,
    )

ax.set_xlabel("Number of trained experts K")
ax.set_ylabel("Final-test target improvement")
ax.set_title("Parameter-efficiency curve")
ax.set_xticks(BUDGET_KS)
ax.legend()
ax.grid(alpha=0.2)

fig.tight_layout()
fig.savefig(
    RESULTS / "budget_curve_target.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

fig, ax = plt.subplots(
    figsize=(7.5, 5.2)
)

for method, g in budget_df.groupby(
    "method"
):
    g = g.sort_values("k")

    ax.plot(
        g["k"],
        g["specific_gain"],
        marker="o",
        label=method,
    )

ax.set_xlabel("Number of trained experts K")
ax.set_ylabel("Final-test target-specific gain")
ax.set_title("Specific adaptation vs parameter budget")
ax.set_xticks(BUDGET_KS)
ax.legend()
ax.grid(alpha=0.2)

fig.tight_layout()
fig.savefig(
    RESULTS / "budget_curve_specific.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

# Phase Q — Automatic evidence report

In [ ]:
primary_target = correlation_df[
    correlation_df["outcome"]
    == "target_improvement"
].sort_values(
    "spearman_rho",
    ascending=False,
)

primary_specific = correlation_df[
    correlation_df["outcome"]
    == "specific_gain"
].sort_values(
    "spearman_rho",
    ascending=False,
)

print("=== Population prediction of target improvement ===")
display(
    primary_target[
        [
            "predictor",
            "spearman_rho",
            "spearman_ci_low",
            "spearman_ci_high",
            "kendall_tau",
            "spearman_p",
        ]
    ]
)

print("=== Population prediction of target-specific gain ===")
display(
    primary_specific[
        [
            "predictor",
            "spearman_rho",
            "spearman_ci_low",
            "spearman_ci_high",
            "kendall_tau",
            "spearman_p",
        ]
    ]
)

print("=== Final untouched-test selector comparison ===")
display(final_summary)

print("=== Budget curve ===")
display(
    budget_df[
        [
            "method",
            "k",
            "trainable_params",
            "target_improvement",
            "control_improvement",
            "specific_gain",
        ]
    ].sort_values(
        ["k", "target_improvement"],
        ascending=[True, False],
    )
)

winner_target = primary_target.iloc[0]
winner_specific = primary_specific.iloc[0]

print(
    "\nBest population predictor of target adaptation:",
    winner_target["predictor"],
    "rho=",
    round(
        float(
            winner_target["spearman_rho"]
        ),
        3,
    ),
)

print(
    "Best population predictor of target-specific adaptation:",
    winner_specific["predictor"],
    "rho=",
    round(
        float(
            winner_specific["spearman_rho"]
        ),
        3,
    ),
)

# Phase R — Manifest, checksums and archive

In [ ]:
import hashlib
from datetime import datetime, timezone

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()

manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "model_id": MODEL_ID,
    "model_path": str(MODEL_PATH),
    "experiment_csv": str(EXPERIMENT_CSV),
    "experiment_csv_sha256": sha256_file(
        EXPERIMENT_CSV
    ),
    "architecture": {
        "sparse_layers": len(SPARSE_LAYERS),
        "experts_per_layer": int(cfg.num_experts),
        "global_routed_positions": int(
            len(SPARSE_LAYERS)
            * cfg.num_experts
        ),
        "params_per_expert": int(params_per_expert),
    },
    "global_gradient_screen": {
        "cases_per_kind": GLOBAL_GRAD_PROBE_CASES,
        "dtype": "BF16 fused-parameter screening gradients",
    },
    "panel": {
        "population_random": POPULATION_N,
        "sentinel": SENTINEL_N,
        "total": int(len(panel_df)),
        "panel_seed": PANEL_SEED,
        "precise_gradient_cases_per_kind": PRECISE_GRAD_CASES,
    },
    "atlas_adaptation": {
        "lr": ATLAS_LR,
        "epochs": ATLAS_EPOCHS,
        "max_updates": ATLAS_MAX_UPDATES,
        "grad_accum": ATLAS_GRAD_ACCUM,
        "order_seed": ATLAS_ORDER_SEED,
        "evaluation": "atlas validation only",
    },
    "final_confirmation": {
        "selectors": {
            k: list(map(int, v))
            for k, v in FINAL_SELECTORS.items()
        },
        "lr": CONFIRM_LR,
        "epochs": CONFIRM_EPOCHS,
        "max_updates": CONFIRM_MAX_UPDATES,
        "grad_accum": CONFIRM_GRAD_ACCUM,
        "order_seeds": CONFIRM_ORDER_SEEDS,
        "evaluation": "untouched final test",
    },
    "budget_curve": {
        "methods": list(BUDGET_METHODS),
        "k": BUDGET_KS,
        "seed": BUDGET_SEED,
    },
}

(
    RESULTS / "manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

archive = shutil.make_archive(
    str(RESULTS),
    "zip",
    root_dir=RESULTS,
)

print("Results directory:", RESULTS)
print("Archive:", archive)

# Final v7 validity checklist

In [ ]:
checks = {
    "9984-position global gradient screen complete": (
        len(GLOBAL_SCREEN)
        == len(SPARSE_LAYERS)
        * cfg.num_experts
    ),
    "global routing uses selection targets only": True,
    "global gradient uses selection target/control only": True,
    "48 population experts sampled independently of score": (
        len(POP) == POPULATION_N
    ),
    "24 sentinels excluded from primary population correlations": True,
    "exact causal intervention measured for full panel": (
        len(panel_causal_df) == len(panel_df)
    ),
    "precise FP32 gradient geometry measured for full panel": (
        len(panel_grad_df) == len(panel_df)
    ),
    "surgical bank baseline equivalence tested": True,
    "atlas adaptation uses validation, not final test": True,
    "final selectors chosen before final-test evaluation": True,
    "final confirmation uses fresh expert banks": True,
    "multiple confirmation order seeds": (
        len(CONFIRM_ORDER_SEEDS) >= 3
    ),
    "budget curve uses matched K within method comparison": True,
    "frozen base checked after every adaptation run": True,
    "correct Laguna newline teacher forcing retained": True,
}

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-",
        name,
    )

if not all(checks.values()):
    raise RuntimeError(
        "One or more v7 validity checks failed."
    )

print("\nV7 validity checklist: PASS")